# 02 — Feature Preparation

This notebook defines the target, excludes identifiers, inspects feature types and establishes a stratified train/test split. Imputation, scaling and encoding are performed inside model pipelines to avoid fitting transformations on held-out data.

In [1]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

ROOT=Path.cwd()
if ROOT.name=='notebooks': ROOT=ROOT.parent
df=pd.read_csv(ROOT/'data/raw/enterprise_saas_churn_synthetic.csv')
TARGET='churned'; ID='account_id'
X=df.drop(columns=[TARGET,ID]); y=df[TARGET].astype(int)
categorical_features=X.select_dtypes(include='object').columns.tolist()
numeric_features=X.select_dtypes(exclude='object').columns.tolist()
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,stratify=y,random_state=42)
print('Rows:',len(df),'Features:',X.shape[1])
print('Train/test:',X_train.shape,X_test.shape)
print('Train/test churn prevalence:',round(y_train.mean(),3),round(y_test.mean(),3))
print('Categorical:',categorical_features)
print('Numeric:',numeric_features)

Rows: 3000 Features: 16
Train/test: (2400, 16) (600, 16)
Train/test churn prevalence: 0.152 0.153
Categorical: ['industry', 'region', 'customer_tier']
Numeric: ['tenure_months', 'annual_recurring_revenue_usd', 'licensed_seats', 'license_utilization_pct', 'dau_mau_ratio', 'active_modules', 'monthly_api_calls', 'qbrs_attended_last_4', 'csat_score', 'kpi_milestones_achieved_pct', 'automated_workflows', 'support_cases_last_90d', 'days_to_renewal']


/tmp/ipykernel_2553/1415727572.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features=X.select_dtypes(include='object').columns.tolist()
/tmp/ipykernel_2553/1415727572.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are excluded by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `exclude` to remove them, or to `include` to keep them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes

## 1. Preprocessing strategy

- Numeric features: median imputation followed by standard scaling.
- Categorical features: most-frequent imputation followed by one-hot encoding.
- Account ID is excluded because it is an identifier, not a predictive signal.
- Stratification preserves the approximate class proportion in train and test sets.
- The holdout set is reserved for final evaluation after cross-validation and tuning.

In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder,StandardScaler

numeric_pipeline=Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())])
categorical_pipeline=Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('encoder',OneHotEncoder(handle_unknown='ignore'))])
preprocessor=ColumnTransformer([('numeric',numeric_pipeline,numeric_features),('categorical',categorical_pipeline,categorical_features)])
print('Preprocessing pipeline configured. It will be fitted only within CV training folds.')

Preprocessing pipeline configured. It will be fitted only within CV training folds.


## 2. Handoff to modeling

Notebook 03 compares Logistic Regression, Decision Tree, Random Forest and HistGradientBoosting. It uses stratified 3-fold cross-validation and GridSearchCV. Average Precision (PR-AUC) is the primary tuning metric; recall, precision, F1 and ROC-AUC provide complementary evaluation.